# BRAIN: synthetic example

This code accompanies *Time is BRAIN: Joint Optimization of Mobile Stroke Unit Deployment and Dispatch to Maximize Patient Health Outcomes* (Markhorst, Jagtenberg, Hov and Larsen), an **unpublished manuscript under journal submission**. See the repository README and `CITATION.cff`.

All locations, counts and travel times below are invented. No study data or study results are included. Install the repository requirements before running this notebook.

In [ ]:
from pathlib import Path
import sys

# Support kernels started in either the repository root or this notebook's folder.
if Path("brain/brain.py").is_file():
    sys.path.insert(0, str(Path("brain").resolve()))

import numpy as np
import pandas as pd
import brain
from IPython.display import display

## Inputs

Rows are origins and columns are destinations; travel times are in **seconds** and can be asymmetric. Stroke counts are annual totals. All matrix locations are candidate MSU bases. Hospitals and ambulance bases use the same location labels. Omit `ambulance_bases` to use the fixed response time in `Parameters`.

BRAIN chooses one base and a maximum dispatch driving time (in **minutes**). The MSU is dispatched only when within that limit and predicted IVT is earlier than conventional care.

In [ ]:
locations = list("ABCD")
travel = pd.DataFrame(
    [[0, 600, 1800, 3000],
     [720, 0, 1200, 2400],
     [1920, 1260, 0, 900],
     [3300, 2520, 960, 0]],
    index=locations, columns=locations,
)
strokes = pd.Series([120., 80., 160., 60.], index=locations)
hospitals = ["D"]
ambulance_bases = ["A", "C"]
radii = [0, 10, 20, 30, 40, 60]
params = brain.Parameters()
display(travel, strokes.to_frame("annual_strokes"))

## Optimize expected outcomes

The default IVT-only planning scenario applies the same outcome curve to all modelled ischemic strokes (86% of total strokes), without a treatment-uptake factor. It estimates annual excellent discharge outcomes (mRS 0–1). This is a modelling assumption, not an estimate of clinical eligibility. The piecewise-linear curve approximates Kim et al. (2017), Figure 2B, between 20 and 270 minutes; its constant continuation beyond those endpoints is an assumption.

All dispatchable confirmed strokes contribute to workload. The service cycle includes outward travel, 30 minutes on scene (including the 22-minute MSU door-to-needle interval), hospital transport, 15-minute handover and return travel. Busy probability is `load / (1 + load)`; blocked calls receive conventional care. Conventional ambulances are assumed available.

In [ ]:
grid = brain.optimize_policy(
    travel, strokes, radii, hospitals, ambulance_bases, params=params,
)
best = grid.iloc[0]
print(f"Best base: {best.base}; maximum dispatch driving time: {best.radius_min:g} min")
display(grid[["base", "radius_min", "expected_outcomes", "additional_outcomes", "busy_probability"]].head())

In [ ]:
benefits = brain.postcode_benefits(
    best.base, best.radius_min, travel, strokes, hospitals, ambulance_bases, params=params,
)
assert len(grid) == len(locations) * len(radii)
np.testing.assert_allclose(benefits.additional_outcomes.sum(), best.additional_outcomes)
display(benefits[["postcode", "dispatchable", "conventional_ott_min", "msu_ott_min", "additional_outcomes"]])

## Extra-call sensitivity

Multipliers increase workload without increasing the outcome population. Extra calls share the stroke geography, dispatch rule and service cycle, and contribute no outcome benefit. Reoptimize at each workload level:

In [ ]:
scenarios = []
for multiplier in [1., 2., 3.]:
    policy = brain.optimize_policy(
        travel, strokes, radii, hospitals, ambulance_bases,
        params=params, call_multiplier=multiplier,
    ).iloc[0].to_dict()
    scenarios.append({"call_multiplier": multiplier, **policy})
display(pd.DataFrame(scenarios)[[
    "call_multiplier", "base", "radius_min", "additional_outcomes", "busy_probability",
]])